# TCL Phase 3 Grounding Runs

Runs inference v2 with the passage in the prompt (and the `--abstain-instruction` line) on the Phase 3 grounding question sets (`data/phase3_grounding/`), then labels every answer's grounding with the NLI model (`label_grounding.py`), as fixed in `docs/phase3/TCL-Phase3-H4-Grounding-Protocol.md`.

One model per session; a model's datasets can be split across sessions with `DATASETS`. Each session writes a zip to `/kaggle/working/`.

## Settings

- Accelerator: **GPU T4 x2**. Internet: **on**.
- `LIMIT = 20` gives a quick smoke run; `None` runs everything.

In [ ]:
# Pin the exact code and data version used for this run.
REPO = "https://github.com/awabmoha/truth-calibration-layer.git"
COMMIT = "c3d34ab85fa0acbea1a9507182f75279b8123a48"

MODEL_TAG = "qwen"   # "qwen" or "phi"
MODELS = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "phi": "microsoft/Phi-3.5-mini-instruct",
}
DATASETS = ["squad", "nq", "hotpotqa", "relation_extraction"]
LIMIT = None
print(MODEL_TAG, MODELS[MODEL_TAG], DATASETS, LIMIT)

In [ ]:
import os, subprocess
if not os.path.exists("truth-calibration-layer"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("truth-calibration-layer")
subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
os.chdir("tcl_experiments")

In [ ]:
!python scripts/check_runtime.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Run

Inference, then grounding labels, per dataset. Re-running this cell resumes finished questions.

In [ ]:
OUT_ROOT = f"/kaggle/working/phase3_grounding/{MODEL_TAG}"
for ds in DATASETS:
    out_dir = f"{OUT_ROOT}/{ds}"
    cmd = [
        "python", "scripts/run_inference_v2.py",
        "--model", MODELS[MODEL_TAG],
        "--model-tag", MODEL_TAG,
        "--dataset", ds,
        "--questions", f"data/phase3_grounding/{ds}.csv",
        "--out-dir", out_dir,
        "--se-splits", "",
        "--abstain-instruction",
        "--resume",
    ]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    subprocess.run(["python", "scripts/label_grounding.py", "--run-dir", out_dir], check=True)

## Package Output

In [ ]:
import shutil
for ds in DATASETS:
    path = f"{OUT_ROOT}/{ds}/records.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            print(ds, sum(1 for _ in f), "records")
tag = "_".join(DATASETS) if len(DATASETS) < 4 else "all"
zip_path = shutil.make_archive(f"/kaggle/working/phase3_grounding_{MODEL_TAG}_{tag}", "zip", OUT_ROOT)
print(zip_path, os.path.getsize(zip_path) // 1_000_000, "MB")